# Inspect dev candidate-bank differences

Compare the strict-extractive 0.5B and 3B candidate banks on the 160-question dev split. This notebook is read-only.

In [ ]:
from pathlib import Path
import json, re, collections
import pandas as pd
import matplotlib.pyplot as plt
PROJECT_ROOT=Path('/home/dinh-tuan/Documents/Project/BioASQ_MRES/Task-Structured Counterfactual Preference Mining')
RUN_ROOT=PROJECT_ROOT/'Artifacts/cse_dpo/candidate_banks/strict_extractive_qwen25_05b_3b_all_splits_s20_v2_seed3407'
BANKS={k:RUN_ROOT/f'qwen25_{k}/dev/adapter-best-evidence-mrr/candidate_bank.jsonl' for k in ('05b','3b')}
for p in BANKS.values(): assert p.exists(), p
print(BANKS)

In [ ]:
def norm(x):
    x=re.sub(r'\[(?:BE|EE)\]',' ',x or '',flags=re.I)
    return re.sub(r'\s+',' ',re.sub(r'[^a-z0-9]+',' ',x.casefold())).strip()
rows={}
for key,path in BANKS.items():
    by_q=collections.defaultdict(list)
    for line in path.open(encoding='utf-8'):
        if line.strip():
            r=json.loads(line); by_q[r['question_id']].append(r)
    rows[key]=by_q
print({k:(len(v),sum(map(len,v.values()))) for k,v in rows.items()})
assert set(rows['05b'])==set(rows['3b'])

In [ ]:
records=[]
for qid in rows['05b']:
    a,b=rows['05b'][qid],rows['3b'][qid]
    sa={norm(r.get('raw_output','')) for r in a if norm(r.get('raw_output',''))}; sb={norm(r.get('raw_output','')) for r in b if norm(r.get('raw_output',''))}
    records.append({'question_id':qid,'question':a[0].get('question_text'),'unique_05b':len(sa),'unique_3b':len(sb),'shared_unique_outputs':len(sa&sb),'only_05b':len(sa-sb),'only_3b':len(sb-sa),'jaccard':len(sa&sb)/len(sa|sb) if sa|sb else 1.0})
df=pd.DataFrame(records)
display(df[['unique_05b','unique_3b','shared_unique_outputs','only_05b','only_3b','jaccard']].describe().round(3))
print('Questions with disagreement:',int((df.only_05b+df.only_3b>0).sum()),'/',len(df))
print('Mean Jaccard:',round(df.jaccard.mean(),3))

In [ ]:
# Show the largest disagreements.
display(df.assign(disagreement=df.only_05b+df.only_3b).sort_values('disagreement',ascending=False).head(25).to_string(index=False))

In [ ]:
# Gold-alias recall: whether any of the 20 samples contains the prepared supported alias.
prepared=PROJECT_ROOT/'Artifacts/Factoid_SFT/models/evidence_grounded_per_supported_alias_qwen25_05b_lora_dropout_005_strict_extractive/trainer_prepared/eval_prepared.json'
gold=collections.defaultdict(set)
for r in json.loads(prepared.read_text()):
    gold[r['id'].split('__',1)[0]].add(norm(r.get('output','')))
recall=[]
for qid in rows['05b']:
    targets=gold.get(qid,set()); result={'question_id':qid,'gold_aliases':sorted(targets)}
    for key in ('05b','3b'):
        outs=[norm(r.get('raw_output','')) for r in rows[key][qid]]
        result[f'{key}_any_sample_exact_gold']=any(x in targets for x in outs)
        result[f'{key}_sample0_exact_gold']=bool(outs) and outs[0] in targets
    recall.append(result)
recall_df=pd.DataFrame(recall)
for key in ('05b','3b'):
    print(key,'any-sample exact gold:',int(recall_df[f'{key}_any_sample_exact_gold'].sum()),'/',len(recall_df),'=',round(recall_df[f'{key}_any_sample_exact_gold'].mean(),4))
    print(key,'sample-0 exact gold:',int(recall_df[f'{key}_sample0_exact_gold'].sum()),'/',len(recall_df),'=',round(recall_df[f'{key}_sample0_exact_gold'].mean(),4))
recall_df.to_csv(RUN_ROOT/'dev_gold_alias_recall.csv',index=False)
print('Saved:',RUN_ROOT/'dev_gold_alias_recall.csv')


In [ ]:
fig,ax=plt.subplots(1,3,figsize=(15,4))
df[['unique_05b','unique_3b']].plot(kind='box',ax=ax[0],title='Unique surfaces/question')
df[['shared_unique_outputs','only_05b','only_3b']].sum().plot(kind='bar',ax=ax[1],title='Aggregate overlap')
df.jaccard.plot(kind='hist',bins=15,ax=ax[2],title='Per-question Jaccard',xlabel='Jaccard')
plt.tight_layout(); plt.show()
out=RUN_ROOT/'dev_bank_difference_summary.csv'; df.to_csv(out,index=False); print('Saved:',out)